# 03a. Input Table Validation and Descriptive Summary


Short summary:
[03a - Validation of Analysis Input Tables (SHORT)](../docs/03a_validate_input_tables_SHORT.md)

## 1. Purpose

This notebook loads and validates the component tables that together form the admission-level analysis dataset used in downstream descriptive and outcome analyses.

Specifically, it inspects:
- A non-ICU adult hospital admission cohort table defined at the hospital admission (HADM) level
- A corresponding early RAAS inhibitor exposure table, also defined at the HADM level and constructed upstream using a fixed early-in-admission exposure window

The primary objective of this notebook is to verify structural integrity and internal consistency prior to analysis. Validation checks include confirmation of a one-to-one correspondence between rows and hospital admissions, alignment of exposure indicators with admission-level identifiers, logical consistency among exposure variables, and the absence of unexpected missingness in key fields.

No tables are created or modified in this notebook; all data are read-only and assumed to have been materialized upstream in BigQuery. The validated tables serve as stable inputs for subsequent baseline characterization and outcome modeling notebooks.

### 1.1 Validation Scope

In observational clinical analyses, careful validation of analytic inputs is essential before performing descriptive summaries or outcome modeling. Errors such as duplicate admissions, unintended row expansion during joins, inconsistent exposure definitions, or silent missingness can introduce bias or invalidate downstream results.

This notebook focuses on validating the admission-level component tables used to define the analytic population and early RAAS inhibitor exposure in a non-ICU cohort derived from the MIMIC-IV database. The cohort table captures demographic, administrative, and hospitalization characteristics at the level of individual hospital admissions, while the exposure table encodes early use of angiotensin-converting enzyme inhibitors and angiotensin receptor blockers based on pre-specified, time-restricted definitions applied upstream.

By systematically inspecting row counts, key identifiers, data types, and logical relationships among exposure indicators, this notebook establishes confidence that the analytic inputs preserve the intended admission-level structure and are suitable for downstream descriptive and outcome analyses.

## 2. Inputs

- **MIMIC-IV v3.1** (BigQuery public dataset)
- Project: `mimic-iv-portfolio`

**Source Tables:**
  - `nonicu_raas.nonicu_admissions`<br>
    (created in [01_cohort.ipynb](01_cohort.ipynb) using [02_exclude_icu_admissions.sql](../sql/02_exclude_icu_admissions.sql))
    
  - `nonicu_raas.exposure_raas_early`<br>
    (created in [02_exposure.ipynb](02_exposure.ipynb) using [03_define_exposure_raas_early.sql](../sql/03_define_exposure_raas_early.sql))

These intermediate tables were generated using SQL-based preprocessing pipelines in BigQuery to ensure reproducibility and a clear separation between data extraction, variable construction, and downstream analysis.

## 3. Cohort Definition

The analytic cohort consists of adult, non-ICU hospital admissions derived from the MIMIC-IV database, with inclusion and exclusion criteria applied upstream using SQL-based data extraction procedures.

Specifically, hospital admissions with any recorded ICU stay were excluded by linking admissions to the MIMIC-IV ICU module and removing all admissions with at least one ICU encounter. As a result, the cohort represents adult non-ICU hospital admissions defined at the hospital admission (HADM) level.

In this notebook, the cohort is represented by a pre-materialized BigQuery table containing one row per hospital admission, uniquely identified by `subject_id` and `hadm_id`. The table is loaded in read-only mode for inspection and validation only; no cohort construction or modification is performed here.

## 4. Exposure Definition

Exposure information was merged into the analytic cohort at the hospital admission level using a left join on `hadm_id` (and `subject_id` as a secondary identifier). Admissions without recorded early RAAS inhibitor exposure were retained and explicitly coded as unexposed, rather than treated as missing.

Exposure status is encoded as admission-level binary indicators for early ACE inhibitor use, early ARB use, combined exposure to both drug classes, and a composite indicator reflecting exposure to either class. These variables were constructed upstream using time-restricted prescription records and are loaded here without modification.

In this notebook, exposure variables are evaluated solely for row alignment, logical consistency (e.g., agreement between component and composite indicators), and completeness, without redefining exposure criteria.

### Workflow boundary

The unified `analysis_dataset` is constructed upstream in [02](02_exposure.ipynb) via [03_build_analysis_dataset.sql](../sql/03_build_analysis_dataset.sql); **03a is read-only and creates no table**. Its checks support the original association branch (03b–04b). They do not validate the separate time-specific risk sets in [04c](04c_prediction_time_evaluation.ipynb), which performs its own ward/ICU chronology and order-availability checks. The original exposure remains prescription records at 0–<24 hours, not confirmed administration. See [current execution evidence](../docs/CLINICAL_REEXECUTION_615de01.md).


## 5. Quality Checks
### 5.1 Dataset Loading and Initial Inspection

## Reading the results

Data sources and execution mode are summarized once below, after the destination settings. Expand **Calculation inputs and source definitions** to trace individual SQL definitions and aggregate references.

Each result is generated by Python in this order: **number/title → figure or aggregate table → how to read → computed result summary → interpretation limits**. Cohort definitions, exposures, adjustment variables and estimators are unchanged. See the [reproduction instructions](../REPRODUCIBILITY.md#python-generated-notebook-results).


The input-definition index describes the Python objects/queries used in calculation. The separate **Saved aggregates and verification files (CSV/JSON)** list identifies saved results and checking records, rather than original MIMIC-IV source tables. Live notebook displays use in-memory results; saved-output replay and validation workflows may read these files.

**Live input behavior:** Normal execution requires the configured real BigQuery inputs and approved access. Missing settings, credentials, permissions or tables stop execution with a cause and a next step; no synthetic/demo data or saved results are substituted. Stored presentation-replay outputs are review artifacts, not a successful new data load. Use a fresh kernel for a new run.


In [ ]:
# Presentation uses only objects produced by this notebook execution.
from pathlib import Path as _ResultPath
import sys as _result_sys
_result_root = _ResultPath.cwd().resolve()
if _result_root.name == "notebooks":
    _result_root = _result_root.parent
if not (_result_root / "scripts" / "nonicu_notebook_results.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory")
if str(_result_root / "scripts") not in _result_sys.path:
    _result_sys.path.insert(0, str(_result_root / "scripts"))
from nonicu_notebook_results import NotebookResults
notebook_results = NotebookResults(globals())


In [ ]:
from real_data_access import real_data_access, real_data_io
from google.cloud import bigquery
from google.auth import default
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Define project ID, dataset, and table references
PROJECT_ID = "mimic-iv-portfolio"
DATASET = "nonicu_raas"

TABLE_NONICU = f"{PROJECT_ID}.{DATASET}.nonicu_admissions"   # Created in 02_exclude_icu_admissions.sql
TABLE_EXPO = f"{PROJECT_ID}.{DATASET}.exposure_raas_early"   # Created in 03_define_exposure_raas_early.sql

# 2. Get ADC credentials

with real_data_access(PROJECT_ID, DATASET):
    creds, adc_project = default()
    client = bigquery.Client(project=PROJECT_ID, credentials=creds)

print("Connected to:", PROJECT_ID, "| ADC default:", adc_project)


In [ ]:
# Shared source references and execution mode for all numbered results.
notebook_results.overview(['Table 03a.1', 'Table 03a.2'])


### 5.2 Dataset Overview and Descriptive Checks

In [ ]:
# Helper for read-only SELECT queries → DataFrame
@real_data_io
def query_to_df(query) :
    """
    Run a SELECT query in BigQuery and return a pandas DataFrame.
    """
    job = client.query(query)
    return job.to_dataframe(create_bqstorage_client=False)

### 5.3 Loading Intermediate BigQuery Tables into pandas DataFrames

In [ ]:
q_nonicu = f"SELECT * FROM `{TABLE_NONICU}`"
q_expo = f"SELECT * FROM `{TABLE_EXPO}`"

df_nonicu = query_to_df(q_nonicu)
df_expo = query_to_df(q_expo)


### 5.4 Schema and Data Type Inspection of the Non-ICU Admissions Table

In [ ]:
with notebook_results.capture('Table 03a.1', globals()):
    df_nonicu.info()
    _notebook_result_value = (print("Shape:", df_nonicu.shape))


### 5.5 Schema and Data Type Inspection of the Early RAAS Exposure Table

In [ ]:
with notebook_results.capture('Table 03a.2', globals()):
    df_expo.info()
    print("Shape:", df_expo.shape)
    _notebook_result_value = (df_expo["raas_any_early"].value_counts(dropna=False))


---
## Next Step

Proceed to:
- [03b. Baseline Characteristics by Early RAAS Exposure](03b_describe_analysis_dataset.ipynb)

This next notebook/document briefly describes the next workflow step.